In [8]:
import lxml.html
import requests
import time
import uuid
import pandas as pd

In [10]:
def make_request(url):
    user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    acc = 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
    headers = {
    'User-Agent': user_agent,
    'Accept': acc,
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    }

    resp = requests.get(url, headers = headers)

    if resp.status_code == 429:
        time.sleep(2.5)
        resp = make_request(url)
    elif resp.status_code == 200:
        return resp

In [11]:
def scrape_judge(url):
    j_page = lxml.html.fromstring(make_request(url).text)
    r_d = {}

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//h3/text()"
    item_titles = j_page.xpath(xp1 + xp2)

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//p/text()"
    items = j_page.xpath(xp1 + xp2)

    for i, title in enumerate(item_titles):
        r_d[title.lower()] = items[i].replace("\t", "").strip().lower()

    return r_d

In [14]:
def scrape_main(url):
    main_page = lxml.html.fromstring(make_request(url).text)

    xp1 = "//section[@filter = 'judge']//"
    xp2 = "a[contains(@href, 'judge')]//@href"
    judge_links = main_page.xpath(xp1 + xp2)

    xp1 = "//section[@filter = 'judge']//a[contains(@href, 'judge')]//"
    xp2 = "div[@class = 'module--content module--content-post']"
    xp3 = "//h2[@class = 'title']/text()"
    judge_names = main_page.xpath(xp1 + xp2 + xp3)

    xp1 = "//div[@class = 'about-icons']"
    xp2 = "//div[@class = 'about-icon' and @data-type = 'state']"
    xp3 = "//span[not(contains(@class, 'sr-only'))]/text()"
    judge_states = main_page.xpath(xp1 + xp2 + xp3)

    judge_fields = [
        "gender",
        "party",
        "race",
        "professional experience",
        "election type",
        "term start",
        "term end",
        "next election date"
    ]

    judge_pd = {
        "JID": [],
        "name": [],
        "state": []
    }

    for field in judge_fields:
        judge_pd[field] = []
    
    for i, name in enumerate(judge_names):
        judge_pd["name"].append(name)
        judge_pd["JID"].append(uuid.uuid4())
        judge_pd["state"].append(judge_states[i])

        judge_dic = scrape_judge(judge_links[i])

        if judge_dic == {}:
            break

        for field in judge_fields:

            if field not in judge_dic:
                data = pd.NA
            else:
                data = judge_dic[field]

            judge_pd[field].append(data)

    return pd.DataFrame(judge_pd)

In [15]:
url = "https://state-law-research.org/state-justices/"

scrape_main(url)

,JID,name,state,gender,party,race,professional experience,election type,term start,term end,next election date
0,7cb33b14-4ed6-4e2c-bef0-8db65f4a89a6,Abigail LeGrow,DE,female,unsure,white,"corporate, judicial","appointed, leg confirmed","may 3, 2023",2035,requires re-nomination and re-confirmation
1,b969843d-227f-489f-9b92-783b36c3028e,Adam Tanenbaum,FL,male,republican,white,"government civil (non civil rights), judicial,...",appointed,"january 14, 2026","january 2, 2029",<NA>
2,ab273ece-c65a-481c-85ee-1e8cda0a6fdd,Aimee A. Oravec,AK,female,unsure,white,"corporate, misc. private practice",appointed,"january 31, 2025",2028,2028
3,19be7666-5a45-4f68-b26b-cbccd362092a,Alisa Kelli Wise,AL,female,republican,white,"corporate, misc. private practice, judicial, o...","elected, partisan",2011,"january 15, 2029","november 2, 2028"
4,55717529-5481-40a1-a571-d20fdc45f523,Allison Riggs,NC,female,democrat,white,"legal aid/non-govt civil rights, judicial",appointed,"september 11, 2023","december 31, 2024","november 7, 2024"
...,...,...,...,...,...,...,...,...,...,...,...
340,e386462d-3bd0-45e4-ab8c-1bf7ff30f307,"William J. Musseman, Jr.",OK,male,republican,white,"prosecutor, judicial",appointed,2022,unsure,unsure
341,66411a20-2b69-4351-accd-0630051edcf0,William P. Robinson III,RI,male,republican,white,misc. private practice,"appointed, leg confirmed",2004,lifetime,n/a - appointed for life
342,29646c7c-93d3-426c-afc5-4786b35eb8a5,William R. Wooton,WV,male,democrat,white,"prosecutor, plaintiff-side civil private pract...","elected, nonpartisan","january 1, 2021",december 2032,tbd
343,ef7e2e28-a9ba-495d-a811-25a4ecc61478,"William W. Hood, III",CO,male,democrat,white,"prosecutor, corporate, plaintiff-side civil pr...","appointed, retention elected","january 13, 2014","january 11, 2027","november 3, 2026"
